<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter8/error_fix_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What to do when you get an error

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

In [5]:
from distutils.dir_util import copy_tree
import os
import tempfile
from huggingface_hub import HfApi, snapshot_download, create_repo, get_full_repo_name

def copy_repo_template():
    api = HfApi()

    template_repo_id = "pop123ux/distilbert-base-uncased-finetuned-squad"
    commit_hash = "xyz"

    template_repo_dir = snapshot_download(template_repo_id, revision=commit_hash)

    model_name = template_repo_id.split("/")[1]

    repo_url = create_repo(model_name, exist_ok=True)
    new_repo_id = repo_url.repo_id

    with tempfile.TemporaryDirectory() as tmp_dir:
        copy_tree(template_repo_dir, tmp_dir)

        for root, dirs, files in os.walk(tmp_dir):
            for file in files:
                if file.startswith('.git') or file == '.nojekyll':
                    try: os.remove(os.path.join(root, file))
                    except: pass

        api.upload_folder(
            folder_path=tmp_dir,
            repo_id=new_repo_id,
            commit_message="Initial commit from template"
        )

In [8]:
from transformers import pipeline

model_checkpoint = get_full_repo_name("distilbert-base-uncased-finetuned-squad")
reader = pipeline("question-answering", model=model_checkpoint)

OSError: pop123ux/distilbert-base-uncased-finetuned-squad-d5716d28 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

A quick way to access the contents of a repo on the HF hub is via the list_repo_files() function of the huggingface_hub library:

In [9]:
from huggingface_hub import list_repo_files

list_repo_files(repo_id=model_checkpoint)

RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-6a551759-0fce3b1900a9ed91296c54df;6929d9af-1d35-4d9b-8403-38df82651581)

Repository Not Found for url: https://huggingface.co/api/models/pop123ux/distilbert-base-uncased-finetuned-squad-d5716d28/tree/main?recursive=true&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication

Interesting -  there doesn't seem to be a config.json file in the repo! No wonder our pipeline couldn't load the model; our collegue must have forgotten to pusht this file to the Hub after they fine-tuned it. In this case, the problem seems pretty straightforward to fix: we could ask them to add the file, or, since we can see from the model ID that the pretrained model used was distil-bert-uncased, we can download the config for this model and push it to our repo to see if that resolves the problem. Let's try that!

In [7]:
from transformers import AutoConfig

pretrained_checkpoint = 'distilbert-base-uncased'
config = AutoConfig.from_pretrained(pretrained_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

KeyboardInterrupt: 

We can then push this to our model repository with the config's push_to_hub() function:

In [ ]:
config.push_to_hub(model_checkpoint, commit_message='Add config.json')

Now we can test if this worked by loading the model from the latest commit on the main branch:

In [ ]:
reader = pipeline('question-answering', model=model_checkpoint, revision='main')

context = r"""Extractive Question Answering is the task of extracting an answer from a text
given a question. An example of a question answering dataset is the SQuAD
dataset, which is entirely based on that task. If you would like to fine-tune a
model on a SQuAD task, you may leverage the
examples/pytorch/question-answering/run_squad.py script.

🤗 Transformers is interoperable with the PyTorch, TensorFlow, and JAX
frameworks, so you can use your favourite tools for a wide variety of tasks!
"""
question = "What is extractive question answering?"
reader(question=question, context=context)

Now let's recap what I have just learned:

- The error messages in Python are known as tracebacks and are read from bottom to top. The last line of the error message usually contains the info we need to locate the source of the problem

- If the last line does not contain sufficient info, we should work our way up the traceback and see if we can identify where in the source code the error occurs

- If none of the error messages can help us debug the problem, we try searching online or asking a LLM

- the huggingface_hub library provides a suite of tools that we can use to interact with and debug repos on the Hub

## Debugging the forward pass of our model ##

Although the pipeline is great for most applications where we need to quickly generate predictions, sometimes we'll need to access the model's logits. To see what could go wrong in this case, let's first grab the model and tokenizer from our pipeline:

In [ ]:
tokenizer=reader.tokenizer
model=reader.model

Next, we need a question, so let's see if our favorite frameworks are supported:

In [6]:
question = "Which frameworks can I use?"

In [10]:
import torch

inputs = tokenizer(question, context, add_special_tokens=True)
input_ids = inputs['input_ids'][0]
outputs = model(**inputs)
answer_start_scores = outputs.start_logits
answer_end_scores = outputs.end_logits

answer_start = torch.argmax(answer_start_scores)
answer_end = torch.argmax(answer_end_scores) + 1
answer = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end])
)
print(f"Question: {question}")
print(f"Answer: {answer}")
